##Pipeline check: does `news_pipeline.py` reproduce our results?

`news_pipeline.py` puts every Tier C rule in one file. This notebook check if the module gives the same results as the notebooks we already ran (Tier_C_FNSPID, Tier_C_ETF_Extraction, Tier_C_Coverage_Audit, TierB_Pilot_Sample, TierB_Gold_Sample), using the existing files in Google Drive.

| Check | Compared with |
|---|---|
| 1. Self-tests (trading days, tickers, relevance, templates) | hand-made expected answers |
| 2. ETF prices: start dates, gaps, backtest window | `etf_prices.csv`, `etf_start_dates.csv`, 30 May 2007 – 29 Dec 2023 |
| 3. S&P 500 list rebuilt from the IVV holdings file | `sp500_sectors.csv` (504 companies) |
| 4. Mapped table and mapping rate | `fnspid_mapped.parquet` (1,321,935 rows), 23.9% |
| 5. Russell 3000 fallback rate | 43.9% |
| 6. News for every ETF and the 50%-of-days rule | `coverage_all_etf_news.csv`: 11 sector ETFs + GLD + AGG |
| 7. Pilot sample drawn again | `pilot_key.csv` (44 headlines) |
| 8. Gold sample and shared subset drawn again | `gold_key.csv` (200 + 100 shared), pool of 369,016 |
| 9. Daily news per sector ETF, 2016–2023 | `coverage_sector_day.csv` |

**Before running:** upload `news_pipeline.py` to **My Drive/pipeline/**.

In [ ]:
import os, sys
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

sys.path.append('/content/drive/MyDrive/pipeline')
assert os.path.exists('/content/drive/MyDrive/pipeline/news_pipeline.py'), 'Upload news_pipeline.py to My Drive/pipeline first'
import news_pipeline as npl

P = npl.paths()
results = []                                                        # one row per check

def record(check, ok, detail=''):
    results.append({'check': check, 'pass': bool(ok), 'detail': detail})
    print(('PASS  ' if ok else 'FAIL  ') + check, '|', detail)

## 1. Self-tests

The six trading-day cases from the ETF notebook (one by one and all at once), ticker spellings on both sides (FNSPID: FB → META, BRK → BRK.B; iShares: BRKB → BRK.B, BF B → BF.B), relevance examples from the pilot, and the extra template patterns.

In [ ]:
cal = npl.load_calendar()
tests = npl.run_self_tests(cal)
record('1. Self-tests', tests['pass'].all(), f"{tests['pass'].sum()} of {len(tests)} pass")
tests

## 2. ETF prices

Start dates, gaps inside each history and the common window, computed by the module from `etf_prices.csv`. Every price date must also be a trading day in the calendar.

In [ ]:
prices = npl.load_prices()
starts_new = npl.start_dates(prices)
starts_saved = pd.read_csv(P['etf_results'] + '/etf_start_dates.csv', index_col=0, parse_dates=['first_date', 'last_date'])
start, end = npl.common_window(prices)
quality = npl.price_quality(prices)

same_starts = (starts_new['first_date'] == starts_saved['first_date'].reindex(starts_new.index)).all()
no_gaps = (quality['gaps_inside_history'] == 0).all()
in_calendar = prices.index.difference(cal.index).empty
record('2. ETF prices', same_starts and no_gaps and in_calendar and (start, end) == (npl.START, npl.END),
       f'window {start.date()} to {end.date()} | start dates match: {same_starts} | no gaps: {no_gaps} | dates in calendar: {in_calendar}')

## 3. S&P 500 list

Rebuilt from `ivv_holdings.csv` with the module and compared company by company with `sp500_sectors.csv` (ticker, company, GICS sector, sector ETF, list date).

In [ ]:
sp500_saved = npl.load_sp500()
sp500_new = npl.build_sp500()
cols = ['ticker', 'company', 'gics_sector', 'sector_etf']
same = sp500_saved[cols].astype(str).reset_index(drop=True).equals(sp500_new[cols].astype(str).reset_index(drop=True))
same_date = str(sp500_new['list_date'][0]) == str(sp500_saved['list_date'][0])
record('3. S&P 500 list', same and same_date and len(sp500_new) == 504,
       f"{len(sp500_new)} companies, list date {sp500_new['list_date'][0]}")

## 4. Mapped table and mapping rate

The clean FNSPID data goes through the module (ticker → S&P 500 company → sector ETF, trading day, backtest window, relevance flag). The result must be identical, row by row, to `fnspid_mapped.parquet`, and the mapping rate must still be 23.9%.

In [ ]:
news = npl.load_clean_news()
sp500 = npl.load_sp500()
mapped_new = npl.build_mapped(news, sp500, cal)
mapped_saved = pd.read_parquet(P['fnspid_mapped'])

cols = ['Date', 'trading_date', 'Article_title', 'Stock_symbol', 'ticker', 'sector_etf', 'relevant']
a = mapped_saved[cols].astype(str).reset_index(drop=True)
b = mapped_new[cols].astype(str).reset_index(drop=True)
record('4a. Mapped table', len(b) == 1_321_935 and a.equals(b), f'{len(b):,} rows (saved: {len(a):,})')

full = npl.map_to_sector_etf(news, sp500)
full['trading_date'] = npl.trading_dates(full['Date'], cal)
window = full[full['trading_date'].between(npl.START, npl.END)]
rate = npl.mapping_rate(window)
record('4b. Mapping rate', rate == 23.9, f'{rate}% of {len(window):,} headlines')

## 5. Russell 3000 fallback

All US companies in the Russell 3000 (iShares IWV holdings, `iwv_holdings.csv` saved by the coverage audit) are mapped through their sector. The fallback rate must still be 43.9%.

In [ ]:
if os.path.exists(P['iwv_holdings']):
    us = npl.build_us_companies()
    broad = npl.map_with_fallback(window, us)
    rate_b = npl.mapping_rate(broad, 'mapped_broad')
    record('5. Fallback rate', rate_b == 43.9, f'{rate_b}% ({len(us):,} US companies)')
else:
    print('Skipped: iwv_holdings.csv not found in etf_extraction/data')

## 6. News for every ETF and the 50%-of-days rule

Own-ticker headlines (e.g. GLD, AGG) and company headlines for all 21 ETFs, as in section 6c of the coverage audit. The table must match `coverage_all_etf_news.csv`, and the ETFs that pass the rule must be the 11 sector ETFs plus GLD and AGG.

In [ ]:
own = npl.own_ticker_news(news, cal)
coverage_new = npl.etf_news_coverage(mapped_new, own, cal)
coverage_saved = pd.read_csv(P['news_results'] + '/coverage_all_etf_news.csv')

cols = ['period', 'etf', 'own_ticker_headlines', 'company_headlines', 'pct_days_with_news', 'signal_possible']
same = coverage_new[cols].astype(str).equals(coverage_saved[cols].astype(str))
signal = npl.news_signal_etfs(coverage_new)
record('6. ETF news coverage', same and sorted(signal) == sorted(npl.SECTOR_ETFS + npl.OWN_NEWS_ETFS),
       f'news signal: {signal}')

## 7. Pilot sample

The pilot was drawn with the first version of the rules (window by calendar year, '-' → '.' only). `pilot_pool` keeps that rule, so the same 44 headlines must come out.

In [ ]:
pilot_saved = pd.read_csv(P['pilot_key'], keep_default_na=False)
pool_pilot = npl.pilot_pool(news, sp500)
pilot_new = npl.sector_sample(pool_pilot, per_sector=4, seed=42, prefix='P')
same = pilot_new[['headline_id', 'Article_title']].astype(str).equals(pilot_saved[['headline_id', 'Article_title']].astype(str))
record('7. Pilot sample', same and len(pool_pilot) == 1_321_949, f'{len(pool_pilot):,} headlines in the pilot pool | 44 IDs and titles match: {same}')

## 8. Gold sample

The annotation pool (2010–2023, informative headlines, no pilot headlines, each title once) and the stratified sample are drawn again with the same seed. The pool must have 369,016 headlines, and the 200 IDs, titles and 100 shared IDs must match `gold_key.csv`.

In [ ]:
pilot = pd.read_csv(P['pilot_key'])                                  # read exactly as the gold notebook did
gold_saved = pd.read_csv(P['gold_key'], keep_default_na=False)

pool = npl.annotation_pool(mapped_saved, years=(2010, 2023), exclude_titles=pilot['Article_title'])
gold_new = npl.stratified_sample(pool, total=200, seed=42)
shared_new = npl.shared_subset(gold_new, size=100, seed=42)

same_titles = gold_new[['headline_id', 'Article_title']].astype(str).equals(gold_saved[['headline_id', 'Article_title']].astype(str))
same_shared = shared_new == set(gold_saved.loc[gold_saved['shared'].astype(str) == 'True', 'headline_id'])
record('8. Gold sample', len(pool) == 369_016 and same_titles and same_shared,
       f'{len(pool):,} in the pool | 200 IDs and titles match: {same_titles} | 100 shared match: {same_shared}')

## 9. Daily news per sector ETF (sector-level aggregation)

`daily_by_etf` gives one row per trading day and one column per sector ETF. For 2016–2023 its median headlines per day and share of days without news must match the coverage audit. In Semester 2 the same function averages headline sentiment into the daily news signal.

In [ ]:
days = cal.index[(cal.index >= '2016-01-01') & (cal.index <= npl.END)]
cov_new = npl.coverage_table(mapped_saved, days)

audit = pd.read_csv(P['news_results'] + '/coverage_sector_day.csv')
audit = audit[(audit['headlines'] == 'all mapped') & (audit['period'] == '2016-2023')].set_index('sector_etf')
same_median = (cov_new['median_per_day'] == audit['median_per_day'].reindex(cov_new.index)).all()
same_days = ((100 - cov_new['pct_days_with_news']) - audit['pct_days_no_news'].reindex(cov_new.index)).abs().max() <= 0.1
record('9. Daily news per sector', same_median and same_days, 'median per day and days without news match the audit')
cov_new

## Summary

In [ ]:
summary = pd.DataFrame(results)
assert summary['pass'].all(), 'At least one check failed: see the table'
print('All checks pass: news_pipeline.py reproduces the existing results.')
summary

##To reuse the modules in other notebooks

In [ ]:
import sys; sys.path.append('/content/drive/MyDrive/pipeline')
import news_pipeline as npl

cal = npl.load_calendar()
news = npl.load_clean_news()
mapped = npl.build_mapped(news, npl.load_sp500(), cal)       # sector ETF + trading day + relevance
usable = npl.keep_informative(mapped)                 # same filter as the gold set
own = npl.own_ticker_news(news, cal, etfs=npl.OWN_NEWS_ETFS)  # GLD and AGG headlines
daily = npl.daily_by_etf(usable, cal.index, value='score')    # Daily sentiment per ETF